Save each trained model's logits on validation, clean test and all 75 CIFAR-10-C sets (Kaggle, 2x T4). Inputs: the cifar10-python dataset, a CIFAR-10-C dataset and the cifar10c-checkpoints dataset. Seeds 0-2 run on GPU 0 and seeds 3-4 on GPU 1 at the same time; the logits are saved to /kaggle/working/outputs/logits.

In [1]:
!nvidia-smi

Thu Nov 19 23:27:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /tmp
![ -d cifar10c-calibration ] || git clone https://github.com/TiaraAnnaJain/cifar10c-calibration.git
%cd /tmp/cifar10c-calibration
!git pull
!pip install -q uv
!uv sync --locked

/tmp
Cloning into 'cifar10c-calibration'...
remote: Enumerating objects: 223, done.
remote: Counting objects: 100% (223/223), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 223 (delta 105), reused 191 (delta 76), pack-reused 0 (from 0)
Receiving objects: 100% (223/223), 122.15 KiB | 11.10 MiB/s, done.
Resolving deltas: 100% (105/105), done.
/tmp/cifar10c-calibration
Already up to date.
Using CPython 3.13.16 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 54 packages in 1ms
⠙ Preparing packages... (0/53)                                                  warning: `build_system.requires = ["uv-build>=0.10.12,<0.11"]` does not contain the current uv version 0.13.9
Prepared 53 packages in 32.01s                                           
Installed 53 packages in 398ms                              
 + ast-serialize==0.13.1
 + cifar10c-calibration==0.1.0 (from file:///tmp/cifar10c-calibration)
 + contourpy==1.4.0
 + cuda-bindings==13.5.0

In [3]:
!mkdir -p /tmp/data/cifar-10-batches-py
!cp $(dirname $(find /kaggle/input -name data_batch_1 | head -1))/* /tmp/data/cifar-10-batches-py/
![ -e /tmp/data/CIFAR-10-C ] || ln -s $(dirname $(find /kaggle/input -name labels.npy | head -1)) /tmp/data/CIFAR-10-C
!ls /tmp/data /tmp/data/CIFAR-10-C | head -30

/tmp/data:
cifar-10-batches-py
CIFAR-10-C

/tmp/data/CIFAR-10-C:
brightness.npy
contrast.npy
defocus_blur.npy
elastic_transform.npy
fog.npy
frost.npy
gaussian_blur.npy
gaussian_noise.npy
glass_blur.npy
impulse_noise.npy
jpeg_compression.npy
labels.npy
motion_blur.npy
pixelate.npy
saturate.npy
shot_noise.npy
snow.npy
spatter.npy
speckle_noise.npy
zoom_blur.npy


In [4]:
!mkdir -p /kaggle/working/outputs/checkpoints
!find /kaggle/input -path "*checkpoints*" -name "resnet18_seed*" -not -path "*fast_dev_run*" -exec cp {} /kaggle/working/outputs/checkpoints/ \;
!ls /kaggle/working/outputs/checkpoints
!grep -h '"test_accuracy"' /kaggle/working/outputs/checkpoints/*.json

resnet18_seed0.json  resnet18_seed2.json  resnet18_seed4.json
resnet18_seed0.pt    resnet18_seed2.pt	  resnet18_seed4.pt
resnet18_seed1.json  resnet18_seed3.json
resnet18_seed1.pt    resnet18_seed3.pt
  "test_accuracy": 0.9386000037193298,
  "test_accuracy": 0.9395999908447266,
  "test_accuracy": 0.9397000074386597,
  "test_accuracy": 0.9369999766349792,
  "test_accuracy": 0.9336000084877014,


In [5]:
check = !uv run python -c "import torch; print(torch.cuda.is_available())"
print(*check, sep="\n")
assert check[-1] == "True", "PyTorch can't see the GPU, so stop here"

Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
True


In [6]:
%cd /tmp/cifar10c-calibration
!CUDA_VISIBLE_DEVICES=0 uv run python scripts/predict.py --seeds 0 1 2 --data-dir /tmp/data --output-dir /kaggle/working/outputs > gpu0.log 2>&1 & \
 CUDA_VISIBLE_DEVICES=1 uv run python scripts/predict.py --seeds 3 4 --data-dir /tmp/data --output-dir /kaggle/working/outputs > gpu1.log 2>&1 & \
 wait
!tail -n 3 gpu0.log gpu1.log

/tmp/cifar10c-calibration
tail: option used in invalid context -- 3


In the saved run, the last line above used `tail -3`, which fails with two files, so the logs weren't printed. It is fixed above; the predictions themselves ran and all 5 logits files were saved (about 11 minutes in total).